# Task 13 — Run it (end-to-end, REAL baseline)

**Wave 3** · target file: `—` · prerequisites: task 07, task 10

**🎯 Goal:** End-to-end smoke with the REAL baseline (no code to write).

Part of the *2026-06-17 fused photo-z backend*. Plan & deps: `../PLAN.md` · conventions: `../README.md`.

In [ ]:
# === Setup: run me first. Hops to the repo root so `import backend` and models/ resolve,
# and pulls the model artifact(s) from GCS if missing (needs gcloud auth - see README). ===
import os, sys
p = os.path.abspath('.')
while p != '/' and not (os.path.isdir(p+'/backend') and os.path.isdir(p+'/notebooks')):
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/fake_image_model.pkl'):
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/fake_image_model.pkl models/fake_image_model.pkl')
if not os.path.exists('models/baseline_stack.pkl'):
    print('pulling the 657MB real baseline (one-time)...')
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/baseline_stack.pkl models/baseline_stack.pkl')
print('models:', [f for f in os.listdir('models') if f.endswith('.pkl')])

## What to do
No code to implement — this proves the assembled API works with the **real** `baseline_stack.pkl`
(the Setup cell pulls the 657MB artifact). Run the cell: `GET /`, an **image-only** prediction (fake
branch), and an **image + tabular** prediction that goes through the **real baseline**. A bright,
low-z-looking galaxy row should come back with a small `z`.

## 🛠️ Develop & test here first
Fill the `# TODO` so the asserts pass. **Don't** paste the answer — write it.

In [ ]:
import os, io, json, numpy as np, importlib
os.environ.pop("BASELINE_PATH", None); os.environ.pop("IMAGE_MODEL_PATH", None)   # use models/ defaults
import backend.config, backend.model, backend.main
for m in (backend.config, backend.model, backend.main): importlib.reload(m)
from fastapi.testclient import TestClient
c = TestClient(backend.main.app)
print("GET / :", c.get("/").json())
def npy(a):
    b = io.BytesIO(); np.save(b, a); b.seek(0); return b
img = np.random.RandomState(0).uniform(0, 5, (64, 64, 5)).astype("float32")
print("image-only          :", c.post("/predict", files={"file": ("c.npy", npy(img))}, data={"ra": 150, "dec": 2}).json())
tab = json.dumps({"dered_u":18.9,"dered_g":17.3,"dered_r":16.6,"dered_i":16.2,"dered_z":15.9,
                  "expRad_r":3.1,"deVRad_r":2.8,"petroRad_r":4.5,"petroR50_r":2.2,"petroR90_r":5.6,"fracDeV_r":0.95})
out = c.post("/predict", files={"file": ("c.npy", npy(img))}, data={"ra": 150, "dec": 2, "tabular": tab}).json()
print("image+tabular (REAL):", out)
assert out["z"] < 0.15, "bright galaxy should be low-z"
print("end-to-end OK")

## ➡️ No file to edit
This task has no `backend/` file — it's the end-to-end smoke. Just make the cell above print **end-to-end OK**.

In [ ]:
print("Nothing to import here - task 13 is the end-to-end smoke above.")
print("If the cell above printed 'end-to-end OK', you are done. Commit your task.ipynb.")

> 💬 *Discuss freely with the team; post anything useful to the Knowledge Base.*

## 🚀 Ship it

In [ ]:
# === Commit & push on YOUR OWN branch (run last; repo root after Setup) ===
!git checkout -B task/13-run-it
!git add notebooks/tasks-2026-6-17/13-run-it/task.ipynb
!git commit -m "13-run-it: Run it (end-to-end, REAL baseline)"
!git push -u origin task/13-run-it
# then merge back into 2026.6.17 -> see MERGE.md in this folder